# Laboratorio 7 – Perfiles y salarios de la población asalariada de la ENEIC con Spark

Este notebook analiza los ingresos laborales de la población asalariada de Guatemala a partir de las bases de Personas de la Encuesta Nacional de Empleo e Ingresos Continua del Instituto Nacional de Estadística. Se utilizan los cuatro trimestres de 2025 para el desarrollo y el primer trimestre de 2026 se prepara y se reserva para la evaluación final del modelado supervisado.

El trabajo se realiza con Spark 3.5 y la API de DataFrames de pyspark.ml. Los archivos originales vienen en formato de hoja de cálculo, por lo que se leen con pandas y openpyxl, se convierten a DataFrames de Spark con tipos explícitos y se almacenan en Parquet. A partir de ese punto toda la preparación analítica, el cálculo de métricas y el aprendizaje automático se realizan en Spark. A pandas solo se transfieren tablas agregadas o muestras acotadas para graficar.

Esta entrega corresponde al avance del laboratorio y cubre cuatro ejercicios. El Ejercicio 1 carga los cinco archivos, identifica su procedencia, homologa tipos, documenta los valores faltantes, aplica una cadena de filtros en un orden fijo, verifica la unicidad de la clave de registro y guarda los conjuntos preparados en Parquet. El Ejercicio 2 describe la población analítica de 2025 mediante estadísticos de posición y dispersión y responde con evidencia gráfica cómo se distribuyen los registros y el salario. El Ejercicio 3 estima la correlación de Pearson entre las variables numéricas con Correlation.corr de pyspark.ml.stat. El Ejercicio 4 construye una segmentación de perfiles con KMeans, compara variantes con y sin salario, evalúa valores de K entre 2 y 5 y describe los grupos obtenidos. Las asociaciones encontradas son descriptivas de los registros analizados y no constituyen relaciones causales ni recomendaciones salariales.

## Verificación de entorno

Antes de cualquier análisis se comprueba la versión de Python en uso, el intérprete que ejecuta el notebook, la versión de Java disponible para Spark, la presencia de los paquetes requeridos y la existencia de las carpetas y de los cinco archivos originales de la encuesta. Si alguno de estos elementos falta, el resto del cuaderno no puede ejecutarse de forma reproducible.

In [ ]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)

# ubicacion de la raiz del proyecto tomando en cuenta que el notebook vive en la carpeta notebook
DIR_BASE = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
print("Raiz del proyecto:", DIR_BASE)

print("-" * 60)

# version de java que utilizara la maquina virtual de spark
ruta_java = shutil.which("java")
if ruta_java is None:
    print("java: no encontrado en el PATH")
else:
    salida_java = subprocess.run([ruta_java, "-version"], capture_output=True, text=True)
    linea_version = [l for l in (salida_java.stderr + salida_java.stdout).splitlines() if "version" in l.lower()]
    print("java:", linea_version[0].strip() if linea_version else "version no identificada")

print("-" * 60)

PAQUETES_REQUERIDOS = ["pyspark", "pandas", "numpy", "openpyxl", "pyarrow", "matplotlib", "seaborn"]
for nombre in PAQUETES_REQUERIDOS:
    try:
        modulo = importlib.import_module(nombre)
        print(f"{nombre:<12}", getattr(modulo, "__version__", "version no expuesta"))
    except ImportError:
        print(f"{nombre:<12}", "NO DISPONIBLE")

## Configuración del proyecto y rutas

Las rutas del proyecto se definen una sola vez y se reutilizan durante todo el cuaderno. Los datos originales permanecen sin modificación en data/raw, los conjuntos derivados se escriben en data/processed y las figuras se exportan a figures con una numeración secuencial que queda registrada en un índice. Esta celda también declara el diccionario de archivos de la encuesta, que asocia cada archivo con el corte publicado al que pertenece.

In [ ]:
import os

DIR_DATOS = DIR_BASE / "data"
DIR_RAW = DIR_DATOS / "raw"
DIR_PROCESADO = DIR_DATOS / "processed"
DIR_FIGURAS = DIR_BASE / "figures"

for carpeta in [DIR_RAW, DIR_PROCESADO, DIR_FIGURAS]:
    carpeta.mkdir(parents=True, exist_ok=True)

INDICE_FIGURAS = DIR_FIGURAS / "indice_figuras.csv"

# cada archivo original se asocia con el corte publicado al que pertenece
ARCHIVOS_ENEIC = {
    "2025T1": {"archivo": "ENEIC 2025 I Personas.xlsx", "anio": 2025, "trimestre": 1, "uso": "entrenamiento"},
    "2025T2": {"archivo": "ENEIC 2025 II Personas.xlsx", "anio": 2025, "trimestre": 2, "uso": "entrenamiento"},
    "2025T3": {"archivo": "ENEIC 2025 III Personas.xlsx", "anio": 2025, "trimestre": 3, "uso": "entrenamiento"},
    "2025T4": {"archivo": "ENEIC 2025 IV Personas.xlsx", "anio": 2025, "trimestre": 4, "uso": "entrenamiento"},
    "2026T1": {"archivo": "ENEIC 2026 I Personas.xlsx", "anio": 2026, "trimestre": 1, "uso": "prueba final"},
}

RUTA_PARQUET_2025 = DIR_PROCESADO / "eneic_2025_analitica.parquet"
RUTA_PARQUET_2026 = DIR_PROCESADO / "eneic_2026_analitica.parquet"

print("Carpeta de datos originales:", DIR_RAW)
print("Carpeta de datos procesados:", DIR_PROCESADO)
print("Carpeta de figuras:", DIR_FIGURAS)

print("-" * 60)

# verificacion de existencia de los archivos originales declarados
for periodo, info in ARCHIVOS_ENEIC.items():
    ruta = DIR_RAW / info["archivo"]
    tamanio = f"{ruta.stat().st_size / 1e6:.1f} MB" if ruta.exists() else "faltante"
    print(f"{periodo}  {info['archivo']:<32} {tamanio}")

### Utilidades de figuras

Las figuras se guardan con un contador secuencial que produce nombres del tipo Figura1.png y registra número, archivo y descripción en indice_figuras.csv. El índice facilita referenciar cada figura en el informe y se regenera desde cero en cada ejecución completa del cuaderno.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12

# el indice se reinicia al definir las utilidades para que una corrida completa lo regenere
if INDICE_FIGURAS.exists():
    INDICE_FIGURAS.unlink()

CONTADOR_FIGURAS = {"n": 0}


def guardar_figura(descripcion):
    CONTADOR_FIGURAS["n"] += 1
    nombre = f"Figura{CONTADOR_FIGURAS['n']}.png"
    ruta = DIR_FIGURAS / nombre
    plt.savefig(ruta, dpi=150, bbox_inches="tight")
    registro = pd.DataFrame([{"numero": CONTADOR_FIGURAS["n"], "archivo": nombre, "descripcion": descripcion}])
    registro.to_csv(INDICE_FIGURAS, mode="a", header=not INDICE_FIGURAS.exists(), index=False)
    return ruta


print("Utilidades de figuras listas")

### Sesión de Spark

Se inicializa una sesión local que utiliza todos los hilos disponibles del procesador y se eleva el tamaño máximo del resultado que puede recibir el driver, porque varias tablas agregadas se materializan localmente para graficar. La conversión con Arrow se deja desactivada de forma deliberada, ya que requiere permisos adicionales de la máquina virtual de Java a partir de la versión 17 y aquí solo se trasladan a pandas tablas agregadas y muestras de a lo sumo diez mil registros, donde no aporta una ventaja apreciable. Con esto el cuaderno corre igual con Java 17 y con versiones posteriores.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("Laboratorio7_ENEIC")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "1g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Java :", spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))
print("Hilos:", spark.sparkContext.defaultParallelism)

## Ejercicio 1 – Carga, armonización y calidad de los datos

Este ejercicio construye la base analítica del laboratorio. Se cargan los cinco archivos de la base de Personas, se identifica el corte publicado del que proviene cada registro, se seleccionan las columnas requeridas y se homologan sus tipos, ya que un mismo código puede llegar como número en un archivo y como texto en otro. Los cuatro archivos de 2025 se apilan con unionByName, se cuantifican los valores faltantes antes de filtrar, se aplica una cadena de filtros en un orden fijo con el conteo de exclusiones en cada paso y se verifica la unicidad de la combinación de periodo de archivo, número de hogar y número de persona. El resultado se guarda en Parquet para 2025 y para 2026 por separado.

Las variables analíticas se derivan de las columnas originales indicadas en el enunciado. La antigüedad en años se construye como antiguedad_anios más antiguedad_meses dividido entre doce. La variable objetivo salario_mensual proviene de P05D01 y corresponde al sueldo o salario mensual sin descuentos de la ocupación principal expresado en quetzales.

### 1.1 – Diccionario de variables y códigos

Se declara la correspondencia entre el nombre original de cada columna y su nombre analítico, junto con las etiquetas de las variables categóricas tomadas del diccionario de datos de la encuesta. Los códigos categóricos se conservan como texto normalizado para evitar que un mismo valor aparezca como 1 en un archivo y como 1.0 en otro. Todo código ausente se representa como DESCONOCIDO y no se convierte a cero, porque en la variable educativa el código 0 significa ninguno y es una categoría válida.

Las etiquetas de la categoría ocupacional están indicadas de forma explícita en el enunciado del laboratorio. Las de nivel educativo y dominio se cargan desde el diccionario de datos que acompaña a cada base. Mientras un diccionario esté vacío, el código se conserva como etiqueta provisional y una celda de diagnóstico lista todos los valores observados con su frecuencia, de modo que completarlos sea un cambio en un solo lugar. Una vez cargado el diccionario, cualquier código que no aparezca en él se marca como DESCONOCIDO, tal como exige el enunciado.

In [ ]:
# correspondencia entre la columna original y el nombre analitico
MAPA_COLUMNAS = {
    "P05D01": "salario_mensual",
    "P02A03": "edad",
    "P05C07A": "antiguedad_anios",
    "P05C07B": "antiguedad_meses",
    "P05H01A": "horas_semanales",
    "P03A03A": "nivel_educativo",
    "P05C16": "categoria_ocupacional",
    "DOMINIO": "dominio",
    "OCUPADOS": "ocupado",
}

# columnas que se conservan con su nombre original para auditoria y documentacion del diseno muestral
COLUMNAS_CONSERVADAS = ["NUM_HOGAR", "NUM_PERSONA", "FACTOR", "ANIO", "TRIMESTRE"]

COLUMNAS_ORIGEN = list(MAPA_COLUMNAS.keys()) + COLUMNAS_CONSERVADAS

# variables que se leen como numero y variables que se leen como codigo de texto
VARIABLES_NUMERICAS = ["salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses", "horas_semanales", "FACTOR"]
VARIABLES_CATEGORICAS = ["nivel_educativo", "categoria_ocupacional", "dominio", "ocupado"]

ETIQUETA_DESCONOCIDA = "DESCONOCIDO"

print("Columnas solicitadas al archivo original:", len(COLUMNAS_ORIGEN))
for original, analitico in MAPA_COLUMNAS.items():
    print(f"{original:<10} {analitico}")

In [ ]:
# etiquetas tomadas del diccionario de datos de la encuesta
ETIQUETAS_CATEGORIA_OCUPACIONAL = {
    "1": "Empleado de gobierno",
    "2": "Empleado de empresa privada",
    "3": "Jornalero o peon",
    "4": "Servicio domestico",
}

# etiquetas del diccionario de datos de la encuesta, se completan con los codigos del archivo de diccionario
ETIQUETAS_NIVEL_EDUCATIVO = {}

ETIQUETAS_DOMINIO = {}

CATEGORIAS_ASALARIADAS = ["1", "2", "3", "4"]

for nombre, etiquetas in [
    ("Categoria ocupacional", ETIQUETAS_CATEGORIA_OCUPACIONAL),
    ("Nivel educativo", ETIQUETAS_NIVEL_EDUCATIVO),
    ("Dominio", ETIQUETAS_DOMINIO),
]:
    print(nombre)
    if not etiquetas:
        print("  diccionario pendiente, se conservara el codigo como etiqueta provisional")
    for codigo, etiqueta in etiquetas.items():
        print(f"  {codigo:<4} {etiqueta}")
    print("-" * 60)

### 1.2 – Conversión de los archivos originales a Parquet

Spark no incorpora un lector nativo de hojas de cálculo, por lo que cada archivo se procesa de forma individual para controlar el consumo de memoria. La lectura se realiza con pandas y openpyxl forzando texto en todas las celdas, lo que evita que el motor de la hoja de cálculo decida tipos distintos para el mismo código en archivos distintos. Enseguida se normalizan los códigos, se convierte el bloque a un DataFrame de Spark con un esquema explícito y se escribe a Parquet. Las columnas de procedencia periodo_archivo, anio_archivo, trimestre_calendario y archivo_origen se agregan en este paso a partir del archivo leído y no a partir de la columna TRIMESTRE, que se conserva intacta para auditoría.

In [ ]:
import numpy as np


def normalizar_codigo(valor):
    """Devuelve el codigo como texto sin decimales superfluos y sin espacios."""
    if valor is None:
        return None
    texto = str(valor).strip()
    if texto == "" or texto.lower() in {"nan", "none", "nat"}:
        return None
    # un mismo codigo puede llegar como 1 o como 1.0 segun el archivo
    try:
        numero = float(texto)
    except ValueError:
        return texto.upper()
    if np.isfinite(numero) and float(numero).is_integer():
        return str(int(numero))
    return texto.upper()


def normalizar_numero(valor):
    """Devuelve el valor como numero de punto flotante o nulo si no es interpretable."""
    if valor is None:
        return None
    texto = str(valor).strip().replace(",", "")
    if texto == "" or texto.lower() in {"nan", "none", "nat"}:
        return None
    try:
        numero = float(texto)
    except ValueError:
        return None
    return numero if np.isfinite(numero) else None


# esquema explicito con el que se construye cada DataFrame de Spark
ESQUEMA_ENEIC = T.StructType(
    [T.StructField(nombre, T.DoubleType(), True) for nombre in
     ["salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses", "horas_semanales", "FACTOR"]]
    + [T.StructField(nombre, T.StringType(), True) for nombre in
       ["nivel_educativo", "categoria_ocupacional", "dominio", "ocupado",
        "NUM_HOGAR", "NUM_PERSONA", "ANIO", "TRIMESTRE",
        "periodo_archivo", "archivo_origen"]]
    + [T.StructField("anio_archivo", T.IntegerType(), True),
       T.StructField("trimestre_calendario", T.IntegerType(), True)]
)

ORDEN_COLUMNAS = [campo.name for campo in ESQUEMA_ENEIC.fields]
print("Esquema analitico con", len(ORDEN_COLUMNAS), "columnas")

In [ ]:
def leer_archivo_eneic(periodo):
    """Lee un archivo original, selecciona columnas, normaliza tipos y devuelve un DataFrame de pandas."""
    info = ARCHIVOS_ENEIC[periodo]
    ruta = DIR_RAW / info["archivo"]
    crudo = pd.read_excel(ruta, dtype=str, engine="openpyxl")
    crudo.columns = [str(c).strip().upper() for c in crudo.columns]

    faltantes = [c for c in COLUMNAS_ORIGEN if c not in crudo.columns]
    if faltantes:
        raise KeyError(f"El archivo {info['archivo']} no contiene las columnas {faltantes}")

    seleccion = crudo[COLUMNAS_ORIGEN].rename(columns=MAPA_COLUMNAS)

    for columna in VARIABLES_NUMERICAS:
        seleccion[columna] = seleccion[columna].map(normalizar_numero).astype("float64")

    for columna in VARIABLES_CATEGORICAS + ["NUM_HOGAR", "NUM_PERSONA", "ANIO", "TRIMESTRE"]:
        seleccion[columna] = seleccion[columna].map(normalizar_codigo).astype("object")

    # la procedencia se asigna a partir del archivo leido y no a partir de la columna TRIMESTRE
    seleccion["periodo_archivo"] = periodo
    seleccion["archivo_origen"] = info["archivo"]
    seleccion["anio_archivo"] = info["anio"]
    seleccion["trimestre_calendario"] = info["trimestre"]

    return seleccion[ORDEN_COLUMNAS], crudo.shape


def convertir_a_parquet(periodo, forzar=False):
    """Convierte un archivo original a Parquet y devuelve la ruta junto con la forma original."""
    destino = DIR_PROCESADO / f"eneic_{periodo}_crudo.parquet"
    if destino.exists() and not forzar:
        filas = spark.read.parquet(str(destino)).count()
        return destino, None, filas
    marco, forma = leer_archivo_eneic(periodo)
    sdf = spark.createDataFrame(marco, schema=ESQUEMA_ENEIC)
    sdf.write.mode("overwrite").parquet(str(destino))
    return destino, forma, marco.shape[0]


REGISTROS_ORIGINALES = {}
resumen_conversion = []

for periodo in ARCHIVOS_ENEIC:
    destino, forma, filas = convertir_a_parquet(periodo)
    REGISTROS_ORIGINALES[periodo] = filas
    resumen_conversion.append(
        {
            "periodo_archivo": periodo,
            "archivo_origen": ARCHIVOS_ENEIC[periodo]["archivo"],
            "uso": ARCHIVOS_ENEIC[periodo]["uso"],
            "filas_originales": filas,
            "columnas_originales": forma[1] if forma is not None else "leido de parquet",
            "parquet": destino.name,
        }
    )

tabla_conversion = pd.DataFrame(resumen_conversion)
tabla_conversion

La tabla anterior confirma la diferencia estructural entre archivos. El corte IV de 2025 se publica con más columnas que los demás cortes, de modo que la misma posición de columna no corresponde a la misma variable en todos los archivos. Este es el motivo por el que la unión se hace por nombre y no por posición.

### 1.3 – Esquema y muestra de las columnas seleccionadas

Se muestra el esquema resultante después de la homologación y cinco registros del conjunto de 2025, de modo que se pueda verificar que los nombres analíticos, los tipos y las columnas de procedencia quedaron como se esperaba.

In [ ]:
def leer_periodos(periodos):
    """Lee los parquet crudos de varios periodos y los apila por nombre de columna."""
    marcos = [spark.read.parquet(str(DIR_PROCESADO / f"eneic_{p}_crudo.parquet")) for p in periodos]
    unido = marcos[0]
    for siguiente in marcos[1:]:
        unido = unido.unionByName(siguiente)
    return unido


PERIODOS_2025 = ["2025T1", "2025T2", "2025T3", "2025T4"]

base_2025 = leer_periodos(PERIODOS_2025).persist()
base_2026 = leer_periodos(["2026T1"]).persist()

print("Registros apilados de 2025:", base_2025.count())
print("Registros de 2026:", base_2026.count())

print("-" * 60)
base_2025.printSchema()

In [ ]:
base_2025.select(
    "periodo_archivo", "anio_archivo", "trimestre_calendario", "ANIO", "TRIMESTRE",
    "NUM_HOGAR", "NUM_PERSONA", "edad", "salario_mensual", "horas_semanales",
    "nivel_educativo", "categoria_ocupacional", "dominio", "ocupado", "FACTOR",
).show(5, truncate=False)

### 1.4 – Correspondencia entre la columna TRIMESTRE y el corte publicado

La columna original TRIMESTRE no coincide con el trimestre calendario del archivo. Se conserva sin modificar y se contrasta con la identificación construida a partir del archivo de procedencia, lo que permite documentar los valores observados sin alterar la respuesta original.

In [ ]:
tabla_trimestres = (
    leer_periodos(PERIODOS_2025 + ["2026T1"])
    .groupBy("periodo_archivo", "anio_archivo", "trimestre_calendario", "ANIO", "TRIMESTRE")
    .count()
    .orderBy("periodo_archivo", "TRIMESTRE")
)
tabla_trimestres.show(50, truncate=False)

### 1.5 – Valores faltantes antes de aplicar los filtros

Se cuantifican los valores ausentes de cada variable seleccionada sobre el conjunto apilado de 2025 y sobre el de 2026, antes de cualquier filtro. Un valor se considera ausente cuando la celda viene vacía o cuando su contenido no puede interpretarse como número en el caso de las variables numéricas. Esta medición se realiza sobre la base completa y no sobre la población analítica, porque muchas de estas columnas solo aplican a las personas ocupadas y asalariadas.

In [ ]:
def tabla_faltantes(sdf, etiqueta):
    """Cantidad y porcentaje de valores nulos por variable seleccionada."""
    columnas = [c for c in ORDEN_COLUMNAS if c not in {"periodo_archivo", "archivo_origen", "anio_archivo", "trimestre_calendario"}]
    total = sdf.count()
    conteos = sdf.agg(*[F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in columnas]).collect()[0].asDict()
    marco = pd.DataFrame(
        [{"variable": c, "faltantes": conteos[c], "porcentaje": round(100 * conteos[c] / total, 2)} for c in columnas]
    )
    marco.insert(0, "conjunto", etiqueta)
    marco["registros"] = total
    return marco.sort_values("faltantes", ascending=False).reset_index(drop=True)


faltantes_2025 = tabla_faltantes(base_2025, "2025 apilado")
faltantes_2026 = tabla_faltantes(base_2026, "2026T1")
pd.concat([faltantes_2025, faltantes_2026], ignore_index=True)

### 1.6 – Construcción de la antigüedad y de las etiquetas categóricas

La antigüedad en años se obtiene sumando el componente de años y el componente de meses dividido entre doce. Los códigos categóricos se traducen a etiquetas legibles y cualquier código ausente o no presente en el diccionario se marca como DESCONOCIDO. Estas transformaciones se aplican de forma idéntica al conjunto de 2025 y al de 2026 mediante una sola función, lo que garantiza que las reglas de preparación sean las mismas para desarrollo y para la prueba final.

In [ ]:
def mapa_a_expresion(columna, etiquetas):
    """Traduce un codigo a su etiqueta y marca como DESCONOCIDO lo ausente o no reconocido."""
    expresion = F.when(F.col(columna).isNull(), F.lit(ETIQUETA_DESCONOCIDA))
    # mientras el diccionario no este cargado se conserva el codigo como etiqueta provisional
    if not etiquetas:
        return expresion.otherwise(F.concat(F.lit("Codigo "), F.col(columna)))
    for codigo, etiqueta in etiquetas.items():
        expresion = expresion.when(F.col(columna) == codigo, F.lit(etiqueta))
    return expresion.otherwise(F.lit(ETIQUETA_DESCONOCIDA))


def preparar_variables(sdf):
    """Construye la antiguedad en anios y las etiquetas categoricas del conjunto."""
    return (
        sdf
        .withColumn("antiguedad", F.col("antiguedad_anios") + F.col("antiguedad_meses") / F.lit(12.0))
        .withColumn("nivel_educativo_etiqueta", mapa_a_expresion("nivel_educativo", ETIQUETAS_NIVEL_EDUCATIVO))
        .withColumn("categoria_ocupacional_etiqueta", mapa_a_expresion("categoria_ocupacional", ETIQUETAS_CATEGORIA_OCUPACIONAL))
        .withColumn("dominio_etiqueta", mapa_a_expresion("dominio", ETIQUETAS_DOMINIO))
    )


preparado_2025 = preparar_variables(base_2025)
preparado_2026 = preparar_variables(base_2026)

preparado_2025.select(
    "edad", "antiguedad_anios", "antiguedad_meses", "antiguedad",
    "nivel_educativo", "nivel_educativo_etiqueta",
    "categoria_ocupacional", "categoria_ocupacional_etiqueta",
    "dominio", "dominio_etiqueta",
).show(5, truncate=False)

In [ ]:
# inventario de los codigos observados en cada variable categorica y su frecuencia
for columna, etiqueta_col in [
    ("nivel_educativo", "nivel_educativo_etiqueta"),
    ("categoria_ocupacional", "categoria_ocupacional_etiqueta"),
    ("dominio", "dominio_etiqueta"),
]:
    print(columna)
    (
        preparado_2025
        .groupBy(columna, etiqueta_col)
        .count()
        .orderBy(F.col(columna).cast("double").asc_nulls_first())
        .show(30, truncate=False)
    )

### 1.7 – Aplicación secuencial de los filtros

Los filtros se aplican siempre en el mismo orden y se contabiliza cuántos registros elimina cada paso. El orden empieza por los criterios que definen la población objetivo, edad, condición de ocupación y categoría ocupacional asalariada, continúa con la variable objetivo y termina con los criterios de consistencia de las variables numéricas del modelo. Los registros que no permiten evaluar un criterio, porque el dato es nulo o no interpretable, se excluyen en el paso correspondiente y quedan contabilizados. No se imputa el salario y no se eliminan salarios por ser extremos.

In [ ]:
def aplicar_filtros(sdf, etiqueta):
    """Aplica la cadena de filtros en orden fijo y devuelve el conjunto filtrado y la bitacora."""
    pasos = [
        ("Edad no nula, finita y mayor o igual a 15",
         F.col("edad").isNotNull() & ~F.isnan("edad") & (F.col("edad") >= 15)),
        ("Persona identificada como ocupada",
         F.col("ocupado") == "1"),
        ("Categoria ocupacional asalariada entre 1 y 4",
         F.col("categoria_ocupacional").isin(CATEGORIAS_ASALARIADAS)),
        ("Salario mensual numerico, finito y estrictamente positivo",
         F.col("salario_mensual").isNotNull() & ~F.isnan("salario_mensual") & (F.col("salario_mensual") > 0)),
        ("Antiguedad en anios no nula y no negativa",
         F.col("antiguedad_anios").isNotNull() & ~F.isnan("antiguedad_anios") & (F.col("antiguedad_anios") >= 0)),
        ("Componente de meses entero entre 0 y 11",
         F.col("antiguedad_meses").isNotNull() & ~F.isnan("antiguedad_meses")
         & (F.col("antiguedad_meses") >= 0) & (F.col("antiguedad_meses") <= 11)
         & (F.col("antiguedad_meses") == F.floor("antiguedad_meses"))),
        ("Antiguedad calculada menor o igual a la edad",
         F.col("antiguedad") <= F.col("edad")),
        ("Horas habituales mayores que cero y menores o iguales a 168",
         F.col("horas_semanales").isNotNull() & ~F.isnan("horas_semanales")
         & (F.col("horas_semanales") > 0) & (F.col("horas_semanales") <= 168)),
    ]

    actual = sdf
    antes = actual.count()
    bitacora = [{"paso": 0, "criterio": "Registros iniciales", "antes": antes, "excluidos": 0, "despues": antes}]

    for numero, (criterio, condicion) in enumerate(pasos, start=1):
        siguiente = actual.filter(condicion)
        despues = siguiente.count()
        bitacora.append(
            {"paso": numero, "criterio": criterio, "antes": antes, "excluidos": antes - despues, "despues": despues}
        )
        actual, antes = siguiente, despues

    marco = pd.DataFrame(bitacora)
    marco.insert(0, "conjunto", etiqueta)
    return actual, marco


analitico_2025, bitacora_2025 = aplicar_filtros(preparado_2025, "2025")
analitico_2026, bitacora_2026 = aplicar_filtros(preparado_2026, "2026T1")

analitico_2025 = analitico_2025.persist()
analitico_2026 = analitico_2026.persist()

pd.concat([bitacora_2025, bitacora_2026], ignore_index=True)

In [ ]:
# registros por archivo antes y despues de los filtros
conteo_antes = (
    leer_periodos(PERIODOS_2025 + ["2026T1"])
    .groupBy("periodo_archivo").count().withColumnRenamed("count", "antes_de_filtros")
)
conteo_despues = (
    analitico_2025.select("periodo_archivo").unionByName(analitico_2026.select("periodo_archivo"))
    .groupBy("periodo_archivo").count().withColumnRenamed("count", "despues_de_filtros")
)

tabla_por_archivo = (
    conteo_antes.join(conteo_despues, on="periodo_archivo", how="left")
    .fillna({"despues_de_filtros": 0})
    .withColumn("retencion_porcentaje", F.round(100 * F.col("despues_de_filtros") / F.col("antes_de_filtros"), 2))
    .orderBy("periodo_archivo")
)
tabla_por_archivo.show(truncate=False)

### 1.8 – Verificación de unicidad de la clave de registro

Se comprueba si la combinación de periodo de archivo, número de hogar y número de persona identifica un único registro dentro de cada corte. Cuando aparecen claves repetidas se distingue entre repeticiones exactas, en las que todas las columnas seleccionadas coinciden, y registros en conflicto, en los que la misma clave presenta valores distintos. No se utiliza dropDuplicates, porque eso ocultaría el problema en lugar de documentarlo.

In [ ]:
def revisar_unicidad(sdf, etiqueta):
    """Distingue entre claves repetidas exactas y claves en conflicto."""
    clave = ["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"]
    repetidas = sdf.groupBy(*clave).agg(F.count(F.lit(1)).alias("repeticiones")).filter(F.col("repeticiones") > 1)
    total_claves_repetidas = repetidas.count()

    if total_claves_repetidas == 0:
        return pd.DataFrame(
            [{"conjunto": etiqueta, "claves_repetidas": 0, "registros_involucrados": 0,
              "claves_repeticion_exacta": 0, "claves_en_conflicto": 0}]
        )

    columnas_comparadas = [c for c in sdf.columns if c not in {"archivo_origen"}]
    detalle = (
        sdf.join(repetidas.select(*clave), on=clave, how="inner")
        .groupBy(*clave)
        .agg(
            F.count(F.lit(1)).alias("repeticiones"),
            F.countDistinct(F.struct(*[F.col(c) for c in columnas_comparadas])).alias("versiones_distintas"),
        )
    )
    resumen = detalle.agg(
        F.count(F.lit(1)).alias("claves_repetidas"),
        F.sum("repeticiones").alias("registros_involucrados"),
        F.sum(F.when(F.col("versiones_distintas") == 1, 1).otherwise(0)).alias("claves_repeticion_exacta"),
        F.sum(F.when(F.col("versiones_distintas") > 1, 1).otherwise(0)).alias("claves_en_conflicto"),
    ).collect()[0].asDict()
    resumen["conjunto"] = etiqueta
    return pd.DataFrame([resumen])[
        ["conjunto", "claves_repetidas", "registros_involucrados", "claves_repeticion_exacta", "claves_en_conflicto"]
    ]


unicidad = pd.concat(
    [
        revisar_unicidad(preparado_2025, "2025 antes de filtros"),
        revisar_unicidad(analitico_2025, "2025 poblacion analitica"),
        revisar_unicidad(preparado_2026, "2026T1 antes de filtros"),
        revisar_unicidad(analitico_2026, "2026T1 poblacion analitica"),
    ],
    ignore_index=True,
)
unicidad

In [ ]:
# comprobacion de personas observadas en mas de un periodo dentro del panel longitudinal de 2025
personas_repetidas = (
    analitico_2025
    .groupBy("NUM_HOGAR", "NUM_PERSONA")
    .agg(F.countDistinct("periodo_archivo").alias("periodos_observados"))
)
distribucion_panel = (
    personas_repetidas.groupBy("periodos_observados").count()
    .withColumnRenamed("count", "claves_hogar_persona")
    .orderBy("periodos_observados")
)
distribucion_panel.show(truncate=False)

### 1.9 – Almacenamiento de los conjuntos preparados

Los dos conjuntos preparados se escriben en Parquet por separado. El de 2025 sirve para el análisis exploratorio, la segmentación y el entrenamiento de los modelos supervisados, y el de 2026 queda reservado para la evaluación final. Guardar en Parquet evita releer las hojas de cálculo en las etapas siguientes y conserva los tipos definidos.

In [ ]:
analitico_2025.write.mode("overwrite").parquet(str(RUTA_PARQUET_2025))
analitico_2026.write.mode("overwrite").parquet(str(RUTA_PARQUET_2026))

print("Conjunto analitico de 2025:", RUTA_PARQUET_2025.name, analitico_2025.count(), "registros")
print("Conjunto analitico de 2026:", RUTA_PARQUET_2026.name, analitico_2026.count(), "registros")
print("Columnas guardadas:", len(analitico_2025.columns))

### 1.10 – Respuestas del ejercicio 1

Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos. El corte IV de 2025 se publica con 302 columnas, mientras que los demás cortes traen 270. Apilar por posición asume que la columna número i significa lo mismo en todos los archivos, y en cuanto un archivo agrega, elimina o reordena columnas esa correspondencia se rompe. El resultado sería una tabla en la que los valores de una variable quedan mezclados con los de otra, un error silencioso que no produce ningún mensaje de falla. Por eso la unión se hace con unionByName, que empareja las columnas por su nombre y no por su orden, después de haber seleccionado y renombrado el mismo conjunto de columnas en cada archivo.

Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una respuesta no registrada. En el primer caso la ausencia es estructural y esperada, ya que el cuestionario aplica saltos de patrón y la pregunta nunca se formuló a esa persona. Por ejemplo, las preguntas de salario y de antigüedad solo se formulan a quienes declararon estar ocupados y en relación de dependencia, de modo que para el resto el vacío es la respuesta correcta y no un defecto de la base. En el segundo caso la pregunta sí correspondía y la respuesta no quedó registrada, ya sea por rechazo, por desconocimiento o por un problema de captura, lo que sí constituye información perdida. La distinción importa porque el vacío estructural se resuelve delimitando bien la población analítica, mientras que la no respuesta afecta la cobertura dentro de esa población y puede introducir sesgo si no ocurre al azar. En este laboratorio las variables de empleo presentan una proporción muy alta de vacíos antes de filtrar, y esa proporción cae al restringir la base a personas ocupadas y asalariadas, lo que confirma que la mayor parte de esos vacíos es estructural.

Por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal. La encuesta tiene un diseño longitudinal con rotación, de modo que una parte de las viviendas se entrevista en varios trimestres consecutivos. La unidad de observación no es la persona sino la combinación de persona y período, y cada una de esas filas aporta una medición legítima en un momento distinto, con posibles cambios de salario, de jornada o de antigüedad. Eliminarlas como duplicados descartaría información válida, reduciría de forma arbitraria el tamaño de la base y sesgaría el análisis hacia las personas que solo aparecen una vez, que son precisamente las que entran o salen del panel. Lo que sí debe ser único es la clave dentro de un mismo corte, y eso es lo que se verifica en la sección 1.8.

Por qué el número de registros de la base filtrada no representa a todos los trabajadores del país. Por dos razones distintas. La primera es de cobertura conceptual, ya que la población analítica se restringe a personas de 15 años o más, ocupadas, asalariadas en las categorías 1 a 4 y con un salario mensual positivo registrado. Quedan fuera los trabajadores por cuenta propia, los patronos, los trabajadores familiares no remunerados, los asalariados que no reportaron salario y quienes reportaron un salario de cero. La segunda es de diseño muestral, ya que la encuesta es una muestra probabilística en la que cada registro representa a un número distinto de personas según su probabilidad de selección, y ese número es el que recoge la columna FACTOR. Al trabajar sin ponderar, los conteos describen a los registros analizados y no a la población guatemalteca. Para producir estimaciones poblacionales habría que expandir con FACTOR, por ejemplo calculando totales como la suma del factor y medias como promedios ponderados por el factor, y acompañar los resultados de medidas de error asociadas al diseño. En este laboratorio FACTOR se conserva por trazabilidad y para dejar documentado ese uso, pero no interviene en el clustering, en los modelos ni en las métricas.

## Ejercicio 2 – Estadística descriptiva y exploración de la población analítica

Este ejercicio describe la población analítica de 2025 que resultó de la cadena de filtros. Primero se calculan, sobre el conjunto completo y no sobre una muestra, la cantidad de observaciones, la media, la mediana, la desviación estándar, el mínimo, el máximo y los percentiles 25, 75 y 95 del salario mensual, la edad, la antigüedad y las horas habituales de trabajo. Después se responde con evidencia gráfica cómo se reparten los registros entre categorías ocupacionales, niveles educativos y dominios, qué forma tiene la distribución del salario, qué distancia hay entre su media y su mediana, cómo cambia el salario mediano según el nivel educativo y la categoría ocupacional, y cómo evolucionan el tamaño de la muestra analítica y el salario mediano entre trimestres. Las figuras se construyen a partir de tablas agregadas calculadas en Spark o de muestras acotadas, mientras que todos los estadísticos reportados provienen del conjunto completo.

### 2.1 – Resumen estadístico de las variables numéricas

Los percentiles se calculan con percentile_approx sobre el conjunto completo de la población analítica de 2025, con una precisión alta para que los valores sean estables.

In [ ]:
VARIABLES_RESUMEN = [
    ("salario_mensual", "Salario mensual en quetzales"),
    ("edad", "Edad en anios"),
    ("antiguedad", "Antiguedad en anios"),
    ("horas_semanales", "Horas habituales por semana"),
]


def resumen_numerico(sdf, variables):
    """Estadisticos de posicion y dispersion calculados sobre el conjunto completo."""
    filas = []
    for columna, descripcion in variables:
        agregado = sdf.agg(
            F.count(columna).alias("n"),
            F.mean(columna).alias("media"),
            F.stddev(columna).alias("desviacion"),
            F.min(columna).alias("minimo"),
            F.max(columna).alias("maximo"),
            F.percentile_approx(columna, [0.25, 0.50, 0.75, 0.95], 100000).alias("percentiles"),
        ).collect()[0]
        p25, mediana, p75, p95 = agregado["percentiles"]
        filas.append(
            {
                "variable": descripcion,
                "n": agregado["n"],
                "media": round(agregado["media"], 2),
                "mediana": round(mediana, 2),
                "desviacion": round(agregado["desviacion"], 2),
                "minimo": round(agregado["minimo"], 2),
                "maximo": round(agregado["maximo"], 2),
                "p25": round(p25, 2),
                "p75": round(p75, 2),
                "p95": round(p95, 2),
            }
        )
    return pd.DataFrame(filas)


resumen_2025 = resumen_numerico(analitico_2025, VARIABLES_RESUMEN)
resumen_2025

### 2.2 – Distribución de los registros entre categorías

Se cuentan los registros de la población analítica por categoría ocupacional, nivel educativo y dominio. Los conteos se calculan en Spark y solo la tabla agregada se traslada a pandas para dibujar.

In [ ]:
def conteo_categoria(sdf, columna):
    """Conteo y participacion porcentual de una variable categorica."""
    total = sdf.count()
    return (
        sdf.groupBy(columna).count()
        .withColumn("porcentaje", F.round(100 * F.col("count") / F.lit(total), 2))
        .orderBy(F.desc("count"))
        .toPandas()
    )


conteo_categoria_ocupacional = conteo_categoria(analitico_2025, "categoria_ocupacional_etiqueta")
conteo_nivel_educativo = conteo_categoria(analitico_2025, "nivel_educativo_etiqueta")
conteo_dominio = conteo_categoria(analitico_2025, "dominio_etiqueta")

display(conteo_categoria_ocupacional)
display(conteo_nivel_educativo)
display(conteo_dominio)

In [ ]:
figura, ejes = plt.subplots(3, 1, figsize=(11, 13))

for eje, tabla, titulo in [
    (ejes[0], conteo_categoria_ocupacional, "Registros por categoria ocupacional"),
    (ejes[1], conteo_nivel_educativo, "Registros por nivel educativo"),
    (ejes[2], conteo_dominio, "Registros por dominio"),
]:
    columna = tabla.columns[0]
    sns.barplot(data=tabla, y=columna, x="count", hue=columna, palette="viridis", legend=False, ax=eje)
    eje.set_title(titulo)
    eje.set_xlabel("Registros")
    eje.set_ylabel("")
    for contenedor in eje.containers:
        eje.bar_label(contenedor, fmt="%.0f", padding=3, fontsize=8)

plt.tight_layout()
guardar_figura("Distribucion de registros por categoria ocupacional, nivel educativo y dominio")
plt.show()

### 2.3 – Forma de la distribución del salario

Se compara la distribución del salario mensual en escala original y en escala logarítmica. La escala logarítmica se utiliza solo como recurso de visualización, mientras que el objetivo del modelado permanece en quetzales sin transformar. Se reportan además la asimetría, la curtosis y la distancia entre la media y la mediana calculadas sobre el conjunto completo.

In [ ]:
forma_salario = analitico_2025.agg(
    F.mean("salario_mensual").alias("media"),
    F.expr("percentile_approx(salario_mensual, 0.5, 100000)").alias("mediana"),
    F.skewness("salario_mensual").alias("asimetria"),
    F.kurtosis("salario_mensual").alias("curtosis"),
).collect()[0].asDict()

forma_salario["diferencia_media_mediana"] = forma_salario["media"] - forma_salario["mediana"]
forma_salario["razon_media_mediana"] = forma_salario["media"] / forma_salario["mediana"]

pd.DataFrame([{k: round(v, 4) for k, v in forma_salario.items()}])

In [ ]:
# muestra acotada unicamente para dibujar la distribucion
muestra_salario = (
    analitico_2025.select("salario_mensual")
    .sample(withReplacement=False, fraction=0.10, seed=42)
    .limit(10000)
    .toPandas()
)
print("Registros de la muestra utilizada para graficar:", len(muestra_salario))

figura, ejes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=muestra_salario, x="salario_mensual", bins=60, kde=True, color="steelblue", ax=ejes[0])
ejes[0].axvline(forma_salario["media"], color="red", linestyle="--", label="Media")
ejes[0].axvline(forma_salario["mediana"], color="black", linestyle=":", label="Mediana")
ejes[0].set_title("Distribucion del salario mensual en escala original")
ejes[0].set_xlabel("Salario mensual en quetzales")
ejes[0].set_ylabel("Registros")
ejes[0].legend()

sns.histplot(data=muestra_salario, x="salario_mensual", bins=60, kde=True, color="darkorange", log_scale=True, ax=ejes[1])
ejes[1].axvline(forma_salario["media"], color="red", linestyle="--", label="Media")
ejes[1].axvline(forma_salario["mediana"], color="black", linestyle=":", label="Mediana")
ejes[1].set_title("Distribucion del salario mensual en escala logaritmica")
ejes[1].set_xlabel("Salario mensual en quetzales, escala logaritmica")
ejes[1].set_ylabel("Registros")
ejes[1].legend()

plt.tight_layout()
guardar_figura("Distribucion del salario mensual en escala original y logaritmica")
plt.show()

### 2.4 – Salario mediano por nivel educativo y por categoría ocupacional

Las medianas por grupo se calculan en Spark sobre todos los registros elegibles y se acompañan del número de observaciones de cada grupo, porque una mediana calculada sobre pocos registros no es comparable con una calculada sobre decenas de miles.

In [ ]:
def mediana_por_grupo(sdf, columna):
    """Mediana, media y tamanio de cada grupo calculados sobre el conjunto completo."""
    return (
        sdf.groupBy(columna)
        .agg(
            F.count(F.lit(1)).alias("n"),
            F.expr("percentile_approx(salario_mensual, 0.5, 100000)").alias("mediana_salario"),
            F.round(F.mean("salario_mensual"), 2).alias("media_salario"),
        )
        .orderBy(F.desc("mediana_salario"))
        .toPandas()
    )


mediana_educativo = mediana_por_grupo(analitico_2025, "nivel_educativo_etiqueta")
mediana_ocupacional = mediana_por_grupo(analitico_2025, "categoria_ocupacional_etiqueta")
mediana_dominio = mediana_por_grupo(analitico_2025, "dominio_etiqueta")

display(mediana_educativo)
display(mediana_ocupacional)
display(mediana_dominio)

In [ ]:
figura, ejes = plt.subplots(3, 1, figsize=(11, 13))

for eje, tabla, titulo in [
    (ejes[0], mediana_educativo, "Salario mediano por nivel educativo"),
    (ejes[1], mediana_ocupacional, "Salario mediano por categoria ocupacional"),
    (ejes[2], mediana_dominio, "Salario mediano por dominio"),
]:
    columna = tabla.columns[0]
    sns.barplot(data=tabla, y=columna, x="mediana_salario", hue=columna, palette="mako", legend=False, ax=eje)
    eje.set_title(titulo)
    eje.set_xlabel("Salario mediano en quetzales")
    eje.set_ylabel("")
    for contenedor in eje.containers:
        eje.bar_label(contenedor, fmt="%.0f", padding=3, fontsize=8)

plt.tight_layout()
guardar_figura("Salario mediano por nivel educativo, categoria ocupacional y dominio")
plt.show()

### 2.5 – Evolución entre trimestres

Se observa cómo cambian el tamaño de la población analítica y el salario mediano entre los cuatro cortes de 2025 y el corte de prueba de 2026. El tamaño de la muestra analítica depende tanto del tamaño del archivo original como de la proporción de personas que cumplen los criterios de la población objetivo.

In [ ]:
evolucion = (
    analitico_2025.select("periodo_archivo", "salario_mensual")
    .unionByName(analitico_2026.select("periodo_archivo", "salario_mensual"))
    .groupBy("periodo_archivo")
    .agg(
        F.count(F.lit(1)).alias("n"),
        F.expr("percentile_approx(salario_mensual, 0.5, 100000)").alias("mediana_salario"),
        F.round(F.mean("salario_mensual"), 2).alias("media_salario"),
    )
    .orderBy("periodo_archivo")
    .toPandas()
)
display(evolucion)

figura, ejes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=evolucion, x="periodo_archivo", y="n", hue="periodo_archivo", palette="crest", legend=False, ax=ejes[0])
ejes[0].set_title("Tamanio de la poblacion analitica por periodo")
ejes[0].set_xlabel("Periodo del archivo")
ejes[0].set_ylabel("Registros analiticos")
for contenedor in ejes[0].containers:
    ejes[0].bar_label(contenedor, fmt="%.0f", padding=3, fontsize=9)

sns.lineplot(data=evolucion, x="periodo_archivo", y="mediana_salario", marker="o", color="darkgreen", label="Mediana", ax=ejes[1])
sns.lineplot(data=evolucion, x="periodo_archivo", y="media_salario", marker="s", color="darkred", label="Media", ax=ejes[1])
ejes[1].set_title("Salario mensual por periodo")
ejes[1].set_xlabel("Periodo del archivo")
ejes[1].set_ylabel("Quetzales")
ejes[1].legend()

plt.tight_layout()
guardar_figura("Tamanio de la poblacion analitica y salario por periodo")
plt.show()

### 2.6 – Interpretación del ejercicio 2

Esta sección se redacta a partir de los resultados de la ejecución del cuaderno y responde las preguntas de exploración del enunciado.

Distribución de los registros entre categorías. Se describe cómo se reparte la población analítica entre categorías ocupacionales, niveles educativos y dominios a partir de la Figura de conteos y de las tablas de participación porcentual.

Simetría de la distribución del salario. Se contrasta la forma observada en la Figura de distribución con el coeficiente de asimetría y la curtosis calculados sobre el conjunto completo.

Diferencia entre la media y la mediana del salario. Se reporta la distancia absoluta y la razón entre ambas y se explica qué implica esa distancia para el modelado con una función de pérdida cuadrática.

Salario mediano por nivel educativo y por categoría ocupacional. Se comparan las medianas por grupo junto con el número de observaciones de cada uno, ya que una mediana calculada sobre pocos registros no es comparable con una calculada sobre decenas de miles.

Evolución entre trimestres. Se describe cómo cambian el tamaño de la población analítica y el salario mediano entre los cuatro cortes de 2025 y el corte de 2026.

## Ejercicio 3 – Correlación de Pearson entre las variables numéricas

Este ejercicio mide la asociación lineal entre el salario mensual, la edad, la antigüedad y las horas habituales de trabajo. Las cuatro variables se combinan en una sola columna vectorial con VectorAssembler y la matriz se obtiene con Correlation.corr de pyspark.ml.stat sobre todos los registros elegibles de 2025, no sobre una muestra. La matriz se presenta con sus etiquetas y se acompaña de un mapa de calor. Conviene recordar que el coeficiente de Pearson mide únicamente relación lineal, por lo que un valor cercano a cero no descarta una relación de otra forma.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

VARIABLES_CORRELACION = ["salario_mensual", "edad", "antiguedad", "horas_semanales"]

ensamblador_correlacion = VectorAssembler(inputCols=VARIABLES_CORRELACION, outputCol="vector_correlacion")
vectorizado = ensamblador_correlacion.transform(analitico_2025.select(*VARIABLES_CORRELACION))

matriz_spark = Correlation.corr(vectorizado, "vector_correlacion", method="pearson").head()[0]
matriz_correlacion = pd.DataFrame(
    matriz_spark.toArray(), index=VARIABLES_CORRELACION, columns=VARIABLES_CORRELACION
)
matriz_correlacion.round(4)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(matriz_correlacion, annot=True, fmt=".3f", cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True)
plt.title("Correlacion de Pearson entre las variables numericas, 2025")
plt.tight_layout()
guardar_figura("Mapa de calor de la correlacion de Pearson entre salario, edad, antiguedad y horas")
plt.show()

In [ ]:
# relacion entre edad y antiguedad sobre una muestra acotada para visualizacion
muestra_relaciones = (
    analitico_2025.select(*VARIABLES_CORRELACION)
    .sample(withReplacement=False, fraction=0.05, seed=42)
    .limit(5000)
    .toPandas()
)
print("Registros de la muestra utilizada para graficar:", len(muestra_relaciones))

figura, ejes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=muestra_relaciones, x="edad", y="antiguedad", alpha=0.25, ax=ejes[0])
ejes[0].set_title("Edad frente a antiguedad")
ejes[0].set_xlabel("Edad en anios")
ejes[0].set_ylabel("Antiguedad en anios")

sns.scatterplot(data=muestra_relaciones, x="horas_semanales", y="salario_mensual", alpha=0.25, color="darkorange", ax=ejes[1])
ejes[1].set_yscale("log")
ejes[1].set_title("Horas habituales frente a salario, escala logaritmica en el eje vertical")
ejes[1].set_xlabel("Horas habituales por semana")
ejes[1].set_ylabel("Salario mensual en quetzales, escala logaritmica")

plt.tight_layout()
guardar_figura("Relacion entre edad y antiguedad y entre horas habituales y salario")
plt.show()

### 3.1 – Interpretación del ejercicio 3

Esta sección se redacta a partir de la matriz de correlación obtenida en la ejecución.

Variables con mayor asociación lineal con el salario. Se identifican a partir de la primera fila de la matriz y se interpreta la magnitud de cada coeficiente, recordando que el coeficiente de Pearson mide únicamente relación lineal y que su valor se ve afectado por la asimetría del salario.

Relación entre edad y antigüedad. Se interpreta el coeficiente correspondiente junto con el diagrama de dispersión, incluyendo la restricción estructural de que la antigüedad no puede superar a la edad.

## Ejercicio 4 – Segmentación de perfiles mediante KMeans

Este ejercicio construye una segmentación de la población asalariada de 2025 para identificar perfiles de trabajadores. La pregunta de negocio pide agrupar por características como edad, antigüedad y jornada habitual, por lo que se evalúan dos variantes. La primera utiliza solo variables de trayectoria y jornada, es decir edad, antigüedad y horas habituales, y deja el salario fuera para poder describir después cómo se remunera cada perfil. La segunda incorpora el salario dentro del espacio de agrupamiento. Ambas variantes se estandarizan con StandardScaler, se evalúan con K igual a 2, 3, 4 y 5, y se comparan mediante el coeficiente de silueta con distancia euclidiana y la suma de distancias cuadradas dentro de los clústeres. Con el criterio elegido se selecciona un valor de K y se describe cada grupo con los promedios de sus variables en unidades originales y con su perfil salarial.

### 4.1 – Construcción de las variantes y estandarización

Las variables se ensamblan con VectorAssembler y se estandarizan restando la media y dividiendo entre la desviación estándar. La estandarización es indispensable porque KMeans utiliza distancia euclidiana y las variables están en unidades muy distintas, de modo que sin escalar el salario dominaría por completo la formación de los grupos.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import StandardScaler

SEMILLA = 42

VARIANTES_CLUSTER = {
    "Sin salario": ["edad", "antiguedad", "horas_semanales"],
    "Con salario": ["edad", "antiguedad", "horas_semanales", "salario_mensual"],
}

base_cluster = analitico_2025.select(
    "periodo_archivo", "edad", "antiguedad", "horas_semanales", "salario_mensual",
    "nivel_educativo_etiqueta", "categoria_ocupacional_etiqueta", "dominio_etiqueta",
).persist()

print("Registros utilizados en la segmentacion:", base_cluster.count())


def preparar_espacio(sdf, variables):
    """Ensambla y estandariza las variables de una variante de segmentacion."""
    ensamblador = VectorAssembler(inputCols=variables, outputCol="vector_bruto")
    escalador = StandardScaler(inputCol="vector_bruto", outputCol="caracteristicas", withMean=True, withStd=True)
    modelo = Pipeline(stages=[ensamblador, escalador]).fit(sdf)
    return modelo, modelo.transform(sdf).select("caracteristicas").persist()


espacios = {}
for nombre, variables in VARIANTES_CLUSTER.items():
    modelo_escalado, datos_escalados = preparar_espacio(base_cluster, variables)
    espacios[nombre] = {"modelo": modelo_escalado, "datos": datos_escalados, "variables": variables}
    print(f"{nombre}: {len(variables)} variables, {datos_escalados.count()} registros")

### 4.2 – Selección del número de clústeres

Para cada variante y cada valor de K se entrena un modelo KMeans con semilla fija y se registran la suma de distancias cuadradas dentro de los clústeres, que se usa como criterio de codo, y el coeficiente de silueta, que evalúa qué tan separados y compactos quedan los grupos. La silueta se calcula sobre una muestra acotada porque su cómputo exacto es cuadrático en el número de registros, mientras que el criterio de codo se calcula sobre el conjunto completo.

In [ ]:
VALORES_K = [2, 3, 4, 5]

evaluador_silueta = ClusteringEvaluator(
    featuresCol="caracteristicas", predictionCol="cluster", metricName="silhouette", distanceMeasure="squaredEuclidean"
)

resultados_k = []
modelos_kmeans = {}

for nombre, espacio in espacios.items():
    datos = espacio["datos"]
    # muestra acotada exclusivamente para el calculo de la silueta
    muestra_silueta = datos.sample(withReplacement=False, fraction=0.10, seed=SEMILLA).limit(10000).persist()
    for k in VALORES_K:
        modelo = KMeans(featuresCol="caracteristicas", predictionCol="cluster", k=k, seed=SEMILLA, maxIter=50).fit(datos)
        inercia = modelo.summary.trainingCost
        silueta = evaluador_silueta.evaluate(modelo.transform(muestra_silueta))
        modelos_kmeans[(nombre, k)] = modelo
        resultados_k.append({"variante": nombre, "k": k, "inercia": inercia, "silueta": silueta,
                             "tamanios": sorted(modelo.summary.clusterSizes, reverse=True)})
    muestra_silueta.unpersist()

tabla_k = pd.DataFrame(resultados_k)
tabla_k[["inercia", "silueta"]] = tabla_k[["inercia", "silueta"]].round(4)
tabla_k

In [ ]:
figura, ejes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(data=tabla_k, x="k", y="inercia", hue="variante", marker="o", ax=ejes[0])
ejes[0].set_title("Criterio de codo, suma de distancias cuadradas dentro de los clusteres")
ejes[0].set_xlabel("Numero de clusteres K")
ejes[0].set_ylabel("Inercia")
ejes[0].set_xticks(VALORES_K)

sns.lineplot(data=tabla_k, x="k", y="silueta", hue="variante", marker="o", ax=ejes[1])
ejes[1].set_title("Coeficiente de silueta")
ejes[1].set_xlabel("Numero de clusteres K")
ejes[1].set_ylabel("Silueta")
ejes[1].set_xticks(VALORES_K)

plt.tight_layout()
guardar_figura("Criterio de codo y coeficiente de silueta por variante y valor de K")
plt.show()

### 4.3 – Perfiles del modelo seleccionado

Con la variante y el valor de K elegidos se asigna el clúster a cada registro y se describe cada grupo mediante el número de observaciones, los promedios de las variables de segmentación en sus unidades originales y el salario mediano. El salario se reporta como característica descriptiva del perfil incluso en la variante que no lo utiliza para agrupar, lo que permite observar cómo se remunera cada perfil sin que el salario haya condicionado la formación de los grupos.

In [ ]:
# criterio de seleccion acordado, se elige la combinacion con mayor coeficiente de silueta
mejor_combinacion = tabla_k.sort_values("silueta", ascending=False).iloc[0]
VARIANTE_ELEGIDA = str(mejor_combinacion["variante"])
K_ELEGIDO = int(mejor_combinacion["k"])

print("Variante seleccionada:", VARIANTE_ELEGIDA)
print("Variables de segmentacion:", VARIANTES_CLUSTER[VARIANTE_ELEGIDA])
print("K seleccionado:", K_ELEGIDO)
print("Silueta:", round(float(mejor_combinacion["silueta"]), 4))

modelo_final = modelos_kmeans[(VARIANTE_ELEGIDA, K_ELEGIDO)]
espacio_final = espacios[VARIANTE_ELEGIDA]

# se vuelve a transformar la base completa para conservar las columnas descriptivas junto al cluster
escalado_completo = espacio_final["modelo"].transform(base_cluster)
asignado = modelo_final.transform(escalado_completo).persist()

perfiles = (
    asignado.groupBy("cluster")
    .agg(
        F.count(F.lit(1)).alias("n"),
        F.round(F.mean("edad"), 1).alias("edad_media"),
        F.round(F.mean("antiguedad"), 1).alias("antiguedad_media"),
        F.round(F.mean("horas_semanales"), 1).alias("horas_medias"),
        F.round(F.mean("salario_mensual"), 0).alias("salario_medio"),
        F.expr("percentile_approx(salario_mensual, 0.5, 100000)").alias("salario_mediano"),
    )
    .orderBy("cluster")
    .toPandas()
)
perfiles["participacion"] = (100 * perfiles["n"] / perfiles["n"].sum()).round(2)
perfiles

In [ ]:
# composicion categorica dominante de cada cluster
def categoria_dominante(columna):
    ventana = (
        asignado.groupBy("cluster", columna).count()
    )
    maximos = ventana.groupBy("cluster").agg(F.max("count").alias("count"))
    return (
        ventana.join(maximos, on=["cluster", "count"], how="inner")
        .withColumnRenamed(columna, f"{columna}_dominante")
        .select("cluster", f"{columna}_dominante", "count")
        .orderBy("cluster")
        .toPandas()
    )


composicion = categoria_dominante("categoria_ocupacional_etiqueta")
composicion = composicion.merge(categoria_dominante("nivel_educativo_etiqueta"), on="cluster", suffixes=("_ocupacion", "_educacion"))
composicion = composicion.merge(categoria_dominante("dominio_etiqueta"), on="cluster")
composicion

In [ ]:
# muestra acotada para visualizar la separacion de los perfiles
muestra_clusters = (
    asignado.select("cluster", "edad", "antiguedad", "horas_semanales", "salario_mensual")
    .sample(withReplacement=False, fraction=0.05, seed=SEMILLA)
    .limit(5000)
    .toPandas()
)
muestra_clusters["cluster"] = muestra_clusters["cluster"].astype(str)
print("Registros de la muestra utilizada para graficar:", len(muestra_clusters))

figura, ejes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(data=muestra_clusters, x="edad", y="antiguedad", hue="cluster", palette="tab10", alpha=0.45, s=18, ax=ejes[0])
ejes[0].set_title("Edad frente a antiguedad por cluster")
ejes[0].set_xlabel("Edad en anios")
ejes[0].set_ylabel("Antiguedad en anios")

sns.scatterplot(data=muestra_clusters, x="edad", y="horas_semanales", hue="cluster", palette="tab10", alpha=0.45, s=18, legend=False, ax=ejes[1])
ejes[1].set_title("Edad frente a horas habituales por cluster")
ejes[1].set_xlabel("Edad en anios")
ejes[1].set_ylabel("Horas habituales por semana")

sns.boxplot(data=muestra_clusters, x="cluster", y="salario_mensual", hue="cluster", palette="tab10", legend=False, ax=ejes[2])
ejes[2].set_yscale("log")
ejes[2].set_title("Salario mensual por cluster, escala logaritmica")
ejes[2].set_xlabel("Cluster")
ejes[2].set_ylabel("Salario mensual en quetzales, escala logaritmica")

plt.tight_layout()
guardar_figura("Perfiles obtenidos con KMeans en el espacio de segmentacion y su salario")
plt.show()

### 4.4 – Descripción de los perfiles

Esta sección se redacta a partir de la tabla de perfiles y de la composición categórica obtenidas en la ejecución.

Conveniencia de incluir el salario en la segmentación. Se compara el coeficiente de silueta de las dos variantes y se discute el efecto de la asimetría del salario sobre la formación de los grupos, así como la ventaja de dejarlo fuera para poder describir después cómo se remunera cada perfil.

Elección del número de clústeres. Se justifica el valor de K seleccionado con el criterio acordado por el grupo, que es el mayor coeficiente de silueta, contrastado con el criterio de codo y con el tamaño de cada grupo.

Descripción de los perfiles. A cada clúster se le asigna un nombre descriptivo construido a partir de sus promedios de edad, antigüedad y jornada, de su categoría ocupacional y nivel educativo dominantes y de su salario mediano.

## Conclusiones

Esta sección se redacta al final, a partir de la ejecución completa del cuaderno, y sintetiza los hallazgos de los cuatro ejercicios del avance sin introducir resultados nuevos.

Calidad y alcance de la base analítica. Retención de registros tras la cadena de filtros, naturaleza de los valores faltantes y consecuencias del diseño longitudinal con rotación sobre la interpretación de los conteos.

Comportamiento del salario. Forma de la distribución, distancia entre media y mediana y diferencias entre niveles educativos, categorías ocupacionales y dominios.

Asociaciones lineales. Qué variables numéricas se asocian con el salario y con qué intensidad, y qué implica ese resultado para el modelado supervisado de la segunda entrega.

Perfiles identificados. Segmentos obtenidos, criterio de selección utilizado y lectura de cada perfil en términos de trayectoria laboral y remuneración.

Limitaciones. Los resultados describen a los registros analizados y no constituyen estimaciones oficiales de la población guatemalteca, ya que el análisis es no ponderado y la población está restringida a asalariados con salario positivo registrado. Las asociaciones encontradas son descriptivas y no demuestran relaciones causales.

In [ ]:
# indice de figuras generado durante la ejecucion
indice = pd.read_csv(INDICE_FIGURAS)
display(indice)

base_cluster.unpersist()
asignado.unpersist()
base_2025.unpersist()
base_2026.unpersist()
spark.stop()
print("Sesion de Spark finalizada")